In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
df = pd.read_pickle('taxonomy2.pkl')
boundaries = df['boundary'].str.split(":", expand=True)
df['discontinuous'] = boundaries[2].notnull()
df = df.drop('boundary', axis=1)
df

,fold,domain,taxid,PDB,tax_domain,discontinuous
0,a.1.1.1,d1ux8a_,1423,1UX8_A,Bacteria,False
1,a.1.1.1,d1dlwa_,5885,1DLW_A,Eukaryota,False
3,a.1.1.1,d1dlya_,3054,1DLY_A,Eukaryota,False
4,a.1.1.1,d1uvxa_,3054,1UVX_A,Eukaryota,False
18,a.1.1.1,d1s56b_,1773,1S56_B,Bacteria,False
...,...,...,...,...,...,...
301627,g.44.1.5,d2l80a_,9606,2L80_A,Eukaryota,False
301673,g.46.1.1,d1jjda_,1131,1JJD_A,Bacteria,False
301698,g.46.1.1,d4mt2a_,10117,4MT2_A,Eukaryota,False
302089,g.54.1.1,d1exka_,562,1EXK_A,Bacteria,False


In [6]:
df = df[~df['discontinuous']]
df

,fold,domain,taxid,PDB,tax_domain,discontinuous
0,a.1.1.1,d1ux8a_,1423,1UX8_A,Bacteria,False
1,a.1.1.1,d1dlwa_,5885,1DLW_A,Eukaryota,False
3,a.1.1.1,d1dlya_,3054,1DLY_A,Eukaryota,False
4,a.1.1.1,d1uvxa_,3054,1UVX_A,Eukaryota,False
18,a.1.1.1,d1s56b_,1773,1S56_B,Bacteria,False
...,...,...,...,...,...,...
301627,g.44.1.5,d2l80a_,9606,2L80_A,Eukaryota,False
301673,g.46.1.1,d1jjda_,1131,1JJD_A,Bacteria,False
301698,g.46.1.1,d4mt2a_,10117,4MT2_A,Eukaryota,False
302089,g.54.1.1,d1exka_,562,1EXK_A,Bacteria,False


In [4]:
MLP_predictions  = np.load("../predictions/MLP_predictions.npz")
HMM_predictions  = np.load("../predictions/HMM_predictions.npz")
PLM_predictions  = np.load("../predictions/PLM_predictions.npz")
PSSM_predictions = np.load("../predictions/PSSM_predictions.npz")
all_chains       = np.load("../predictions/all_chains.npz")

In [8]:
AA_dict = {i:j for i,j in zip(['a','r','n','d','c','q','e','g','h','i','l','k','m','f','p','s','t','w','y','v','x'],range(21))}
AA_dict['X'] = -9

astral_file = open("./data/astral-scopedom-seqres-gd-all-2.08-stable.fa", "r")
domain_dict = {}

line = next(astral_file)
dom_name = line.split()[0][1:]
seq = ""
domain_list = list(df['domain'])

for line in astral_file:
    if line[0] == ">":
        if dom_name in domain_list:
            try:
                domain_dict[dom_name] = np.array([AA_dict[x] for x in seq], dtype=np.int8)
                seq = ""
                dom_name = line.split()[0][1:]
            except Exception as e:
                print(dom_name, e, seq)
                seq = ""
                dom_name = line.split()[0][1:]
        else:
            seq = ""
            dom_name = line.split()[0][1:]
    else:
        seq = seq + line.strip()

astral_file.close()

d2fmda_ 'b' ansvcftftdfesgqqdlifqgdasvgsnkalqltkvdskgnpqggsvgralytapirlwqssslvasfettftfsisqgsstpadaltffiaspdtkipsgsggrllglfgssnnagsdngvvsvefdtypntdigdpnyrhigidvnsirskaaskwdwqngktatahisynsaskrlsvvssypnsspvvvsfdvelnnvbpbwvrvgfsattgqytqtnnilawsfrsslmg
d1kp0a2 'z' iksseezblirzgarisdiggaataaaisagvpeyevaiattbamvrziarbfpyvelmdtwiwfqsgintdgahnpvtbrvvzrgdilslncfpmifgyytalertlflzzvbdaslziwzkntavhrrglzlikpgarckdiaselnbmyrzwdllryrtfgyghsfgvlbhyygreagvelrediztvlepgmvvsmepmvmbpegepgaggyrehdilvikenbtenitgfpfgpehniika


In [11]:
df.drop(df[df['domain'].isin(['d2fmda_','d1kp0a2'])].index, inplace=True)
df

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_3612\4051422636.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(df[df['domain'].isin(['d2fmda_','d1kp0a2'])].index, inplace=True)


,fold,domain,taxid,PDB,tax_domain,discontinuous
0,a.1.1.1,d1ux8a_,1423,1UX8_A,Bacteria,False
1,a.1.1.1,d1dlwa_,5885,1DLW_A,Eukaryota,False
3,a.1.1.1,d1dlya_,3054,1DLY_A,Eukaryota,False
4,a.1.1.1,d1uvxa_,3054,1UVX_A,Eukaryota,False
18,a.1.1.1,d1s56b_,1773,1S56_B,Bacteria,False
...,...,...,...,...,...,...
301627,g.44.1.5,d2l80a_,9606,2L80_A,Eukaryota,False
301673,g.46.1.1,d1jjda_,1131,1JJD_A,Bacteria,False
301698,g.46.1.1,d4mt2a_,10117,4MT2_A,Eukaryota,False
302089,g.54.1.1,d1exka_,562,1EXK_A,Bacteria,False


In [12]:
import numba as nb

@nb.jit
def _is_equal_nb(seq, subseq, m, i):
    for j in range(m):
        if seq[i + j] != subseq[j]:
            return False
    return True


@nb.jit
def find_loop_nb(seq, subseq):
    n = len(seq)
    m = len(subseq)
    for i in range(n - m + 1):
        if _is_equal_nb(seq, subseq, m, i):
            yield i

In [13]:
def get_subchain_accuracy(stride, domain_len, prediction, ground_truth):
    prediction = prediction[stride:stride+domain_len]
    ground_truth = ground_truth[stride:stride+domain_len]
    mask = ground_truth!=-9
    prediction = prediction[mask]
    ground_truth = ground_truth[mask]
    assert len(prediction)==len(ground_truth)
    return (sum(prediction==ground_truth)/len(prediction))

def get_discontinuous_accuracy(chain, subchain, prediction, ground_truth):
    split = np.where(subchain==-9)
    split = split[0]
    subchains = np.split(subchain, split)
    subchains[1] = subchains[1][1:]
    stride = next(find_loop_nb(chain, subchains[0]))
    acc1 = get_subchain_accuracy(stride, len(subchains[0]), prediction, ground_truth) * len(subchains[0])
    stride = next(find_loop_nb(chain, subchains[1]))
    acc2 = get_subchain_accuracy(stride, len(subchains[1]), prediction, ground_truth) * len(subchains[1])
    return (acc1+acc2)/(len(subchains[0])+len(subchains[1]))

In [17]:
df['MLP'] = None
df['PLM'] = None
df['PSSM']= None
df['HMM'] = None

for row in df.index:
    PDB_ID = df.loc[row, 'PDB']
    dom_ID = df.loc[row, 'domain']
    
    if dom_ID in domain_dict.keys():
        if PDB_ID in all_chains.files:
            try:
                chain_seq = all_chains[PDB_ID][0]
                domain_seq= domain_dict[dom_ID]
                stride = next(find_loop_nb(chain_seq, domain_seq))

                domain_len = len(domain_seq)
                ground_truth = all_chains[PDB_ID][1]
                df.at[row, 'MLP'] = get_subchain_accuracy(stride, domain_len, MLP_predictions[PDB_ID], ground_truth)
                df.at[row, 'HMM'] = get_subchain_accuracy(stride, domain_len, HMM_predictions[PDB_ID], ground_truth)
                df.at[row, 'PLM'] = get_subchain_accuracy(stride, domain_len, PLM_predictions[PDB_ID], ground_truth)
                df.at[row, 'PSSM']= get_subchain_accuracy(stride, domain_len, PSSM_predictions[PDB_ID], ground_truth)
            except Exception as e:
                pass
#                 print(PDB_ID, e)
                
# for ind in df[df['discontinuous']].index:
#     PDB_ID = df.loc[ind, 'PDB']
#     domain = df.loc[ind, 'domain']
#     try:
#         main_chain = all_chains[PDB_ID][0]
#         sub_seq    = dom_dict[domain]
#         ground_truth = all_chains[PDB_ID][1]

#         prediction = HMM_predictions[PDB_ID]
#         accuracy = get_discontinuous_accuracy(main_chain, sub_seq, prediction, ground_truth)

#         prediction = MLP_predictions[PDB_ID]
#         accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
        
#         prediction = PLM_predictions[PDB_ID]
#         accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
        
#         prediction = PSSM_predictions[PDB_ID]
#         accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
        
#     except Exception as e:
#         print(e)
#         pass

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_3612\445469234.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['MLP'] = None
C:\Users\Mohan2\AppData\Local\Temp\ipykernel_3612\445469234.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['PLM'] = None
C:\Users\Mohan2\AppData\Local\Temp\ipykernel_3612\445469234.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [19]:
df = df[df['PSSM'].notnull()]

#### keep families that still have entries from bacteria and eukaryote

In [24]:
drop_families = []
for family in df['fold']:
    if len(df[df['fold']==family]['tax_domain'].unique()) < 2:
        drop_families.append(family)

In [30]:
drop_index = df[df['fold'].isin(drop_families)].index
df.drop(drop_index, inplace=True)
df

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_3612\2788787342.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(drop_index, inplace=True)


,fold,domain,taxid,PDB,tax_domain,discontinuous,MLP,PLM,PSSM,HMM
0,a.1.1.1,d1ux8a_,1423,1UX8_A,Bacteria,False,0.789916,0.89916,0.806723,0.773109
1,a.1.1.1,d1dlwa_,5885,1DLW_A,Eukaryota,False,0.706897,0.931034,0.818966,0.603448
3,a.1.1.1,d1dlya_,3054,1DLY_A,Eukaryota,False,0.785124,0.917355,0.752066,0.719008
4,a.1.1.1,d1uvxa_,3054,1UVX_A,Eukaryota,False,0.752066,0.92562,0.760331,0.710744
18,a.1.1.1,d1s56b_,1773,1S56_B,Bacteria,False,0.674074,0.925926,0.785185,0.511111
...,...,...,...,...,...,...,...,...,...,...
300853,g.41.5.1,d2v3bb_,208964,2V3B_B,Bacteria,False,0.54717,0.90566,0.660377,0.566038
301673,g.46.1.1,d1jjda_,1131,1JJD_A,Bacteria,False,0.788462,0.884615,0.846154,0.461538
301698,g.46.1.1,d4mt2a_,10117,4MT2_A,Eukaryota,False,0.967213,1.0,0.901639,0.901639
302089,g.54.1.1,d1exka_,562,1EXK_A,Bacteria,False,0.683544,0.759494,0.708861,0.56962


In [31]:
df.to_csv("../results/data/taxonomy_BE.csv")